In [ ]:
# @title
import os
import pandas as pd

# Base folder in Drive
base_path = os.getcwd()

# Weather and Flights subfolders
weather_folder = os.path.join(base_path, 'Weather')
flights_folder = os.path.join(base_path, 'Flights')

# Tolerant CSV reader
def read_csv_tolerant(path):
    return pd.read_csv(path, dtype=str, engine='python', on_bad_lines='skip')

# Read all CSVs from a folder
def read_all_csvs(folder):
    files = [os.path.join(folder, f) for f in os.listdir(folder) if f.endswith('.csv')]
    parts = [read_csv_tolerant(f) for f in files]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

# Load data
weather_df = read_all_csvs(weather_folder)
flights_df = read_all_csvs(flights_folder)

# Clean station / origin codes
weather_df['station'] = weather_df['station'].astype(str).str.strip().str.upper()
flights_df['ORIGIN'] = flights_df['ORIGIN'].astype(str).str.strip().str.upper()

# Parse times
weather_df['valid'] = pd.to_datetime(weather_df.get('valid'), errors='coerce')
flights_df['FlightPlannedDateandTime'] = pd.to_datetime(
    flights_df.get('FlightPlannedDateandTime'),
    errors='coerce'
)

# Drop rows missing key info
weather_df = weather_df.dropna(subset=['station', 'valid'])
flights_df = flights_df.dropna(subset=['ORIGIN', 'FlightPlannedDateandTime'])

# Merge by station with nearest time (no tolerance)
def merge_nearest_by_station_no_tol(flights, weather):
    parts = []
    common = sorted(set(flights['ORIGIN'].unique()) & set(weather['station'].unique()))
    for st in common:
        f = flights.loc[flights['ORIGIN'] == st].sort_values('FlightPlannedDateandTime')
        w = weather.loc[weather['station'] == st].sort_values('valid')
        if len(f) and len(w):
            m = pd.merge_asof(
                f, w,
                left_on='FlightPlannedDateandTime',
                right_on='valid',
                direction='nearest'
            )
            parts.append(m)
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

merged_df = merge_nearest_by_station_no_tol(flights_df, weather_df)

# Save outputs
merged_out = os.path.join(base_path, 'Merged_Flights_Weather.csv')
merged_df.to_csv(merged_out, index=False, na_rep='null')